In [70]:
import nest_asyncio
nest_asyncio.apply()

In [49]:
import os
import asyncio
import logging

# from raganything import RAGAnything
from lightrag import LightRAG
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc, setup_logger,logger ,wrap_embedding_func_with_attrs, set_verbose_debug

In [50]:
from dotenv import load_dotenv

load_dotenv("../.env", override=True)

False

In [51]:
# --------------------------------------------------
# Logger
# --------------------------------------------------
setup_logger("lightrag", level="INFO")

# --------------------------------------------------
# Config
# --------------------------------------------------
WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

In [52]:
# LLM Model Function
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    return await openai_complete_if_cache(
        os.getenv("LLM_MODEL"),
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("LLM_BINDING_API_KEY"),
        base_url=os.getenv("LLM_BINDING_HOST"),
        **kwargs,
    )

async def print_stream(stream):
    async for chunk in stream:
        if chunk:
            print(chunk, end="", flush=True)

In [53]:
async def initialize_rag():

    embedding_dim = int(os.getenv("EMBEDDING_DIM", 1536))
    token_limit = int(os.getenv("EMBEDDING_TOKEN_LIMIT", 8192))
    model_name = os.getenv("EMBEDDING_MODEL")

    # Step 1: define raw embedding function
    async def raw_embedding_func(texts):
        return await openai_embed.func(
            texts,
            api_key=os.getenv("LLM_BINDING_API_KEY"),
            base_url=os.getenv("LLM_BINDING_HOST"),
            model=model_name,
        )

    # Step 2: wrap embedding function
    embedding_func = wrap_embedding_func_with_attrs(
        embedding_dim=embedding_dim,
        max_token_size=token_limit,
        model_name=model_name,
    )(raw_embedding_func)

    # Step 3: initialize LightRAG
    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_model_func,
        embedding_func=embedding_func,
        kv_storage="PGKVStorage",
        vector_storage="PGVectorStorage",
        graph_storage="Neo4JStorage",
    )

    await rag.initialize_storages()
    return rag


In [54]:
# Only run if want to Clear old data files
files_to_delete = [
    "graph_chunk_entity_relation.graphml",
    "kv_store_doc_status.json",
    "kv_store_full_docs.json",
    "kv_store_text_chunks.json",
    "vdb_chunks.json",
    "vdb_entities.json",
    "vdb_relationships.json",
]

for file in files_to_delete:
    file_path = os.path.join(WORKING_DIR, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleting old file:: {file_path}")

In [55]:
def configure_logging():
    """Configure logging for the application"""

    # Reset any existing handlers to ensure clean configuration
    for logger_name in ["uvicorn", "uvicorn.access", "uvicorn.error", "lightrag"]:
        logger_instance = logging.getLogger(logger_name)
        logger_instance.handlers = []
        logger_instance.filters = []

    # Get log directory path from environment variable or use current directory
    log_dir = os.getenv("LOG_DIR", os.getcwd())
    log_file_path = os.path.abspath(
        os.path.join(log_dir, "lightrag_docs_ingestion.log")
    )

    print(f"\nLightRAG docs ingestion log file: {log_file_path}\n")
    os.makedirs(os.path.dirname(log_dir), exist_ok=True)

    # Get log file max size and backup count from environment variables
    log_max_bytes = int(os.getenv("LOG_MAX_BYTES", 10485760))  # Default 10MB
    log_backup_count = int(os.getenv("LOG_BACKUP_COUNT", 5))  # Default 5 backups

    logging.config.dictConfig(
        {
            "version": 1,
            "disable_existing_loggers": False,
            "formatters": {
                "default": {
                    "format": "%(levelname)s: %(message)s",
                },
                "detailed": {
                    "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
                },
            },
            "handlers": {
                "console": {
                    "formatter": "default",
                    "class": "logging.StreamHandler",
                    "stream": "ext://sys.stderr",
                },
                "file": {
                    "formatter": "detailed",
                    "class": "logging.handlers.RotatingFileHandler",
                    "filename": log_file_path,
                    "maxBytes": log_max_bytes,
                    "backupCount": log_backup_count,
                    "encoding": "utf-8",
                },
            },
            "loggers": {
                "lightrag": {
                    "handlers": ["console", "file"],
                    "level": "INFO",
                    "propagate": False,
                },
            },
        }
    )

    # Set the logger level to INFO
    logger.setLevel(logging.INFO)
    # Enable verbose debug if needed
    set_verbose_debug(os.getenv("VERBOSE_DEBUG", "false").lower() == "true")

## Documents

In [56]:
skj_path = "../data/skj_documents"
permenpan_path = "../Data/Documents_pemerintah"
makalah_path = "../Data/makalah"

docs_paths = [skj_path, permenpan_path, makalah_path]

for path in docs_paths:
    if not os.path.exists(path):
        print(f"Warning: The path '{path}' does not exist. Please check the path and try again.")
    else:
        print(f"The path '{path}' exists and is ready for document ingestion.")

The path '../data/skj_documents' exists and is ready for document ingestion.


#### Reader

In [57]:
# Used
import zipfile
from xml.etree import ElementTree as ET
from pathlib import Path
import pdfplumber

NS = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
VMERGE = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val'

def _read_docx(filepath):
    with zipfile.ZipFile(filepath) as z:
        root = ET.fromstring(z.read('word/document.xml'))
    lines = []
    for child in root.find('.//w:body', NS):
        tag = child.tag.split('}')[-1]
        if tag == 'p':
            text = ''.join(
                r.find('w:t', NS).text for r in child.findall('.//w:r', NS)
                if r.find('w:t', NS) is not None and r.find('w:t', NS).text
            ).strip()
            if text:
                lines.append(text)
        elif tag == 'tbl':
            for row in child.findall('w:tr', NS):
                cells = []
                for cell in row.findall('w:tc', NS):
                    vm = cell.find('.//w:vMerge', NS)
                    if vm is not None and vm.get(VMERGE) != 'restart':
                        continue
                    text = ''.join(
                        r.find('w:t', NS).text
                        for p in cell.findall('.//w:p', NS)
                        for r in p.findall('.//w:r', NS)
                        if r.find('w:t', NS) is not None and r.find('w:t', NS).text
                    ).strip()
                    if text:
                        cells.append(text)
                if cells:
                    lines.append(' | '.join(cells))
    return '\n'.join(lines)

def _read_pdf(filepath):
    with pdfplumber.open(filepath) as pdf:
        return '\n'.join(
            page.extract_text() for page in pdf.pages
            if page.extract_text()
        )

def read_folder(folder_path):
    results = []
    for path in Path(folder_path).rglob('*'):
        suffix = path.suffix.lower()
        if suffix == '.docx':
            text = _read_docx(path)
        elif suffix == '.pdf':
            text = _read_pdf(path)
        else:
            continue
        results.append({'filename': path.name, 'text': text})
        print(f"✓ {path.name} ({len(text)} karakter)")
    return results


### Ingestion skj

In [58]:
docs = read_folder("../Data/skj_documents")  # ganti dengan path folder Anda
for doc in docs:
    print(doc['filename'], '-', len(doc['text']), 'karakter')
    print(doc['text'])

✓ 1. Form SKJ Pimpinan Tinggi Madya Sekretaris Utama (Level 5).docx (11372 karakter)
✓ 10. Draft_SKJ_Inspektur_alx_edit - Inspektur 2 v2.docx (11056 karakter)
✓ 11. Form SKJ JPT Pratama (Kepala Biro Umum)_rev1.docx (10188 karakter)
✓ 12. Form SKJ Administrator (Kabag PBJ)_rev1.docx (8224 karakter)
✓ 13. Form SKJ Administrator (Kabag PBMN)_rev1.docx (8572 karakter)
✓ 14. Form SKJ Administrator (Kabag Rumah Tangga)_rev1.docx (8781 karakter)
✓ 15. Form SKJ Administrator (Kabag PKP)_rev1.docx (8488 karakter)
✓ 16. Form SKJ Pengawas (Kasubag Protokol)_Rev.docx (7343 karakter)
✓ 17. Form SKJ Pengawas (Kasubag Kesekretariatan Kepala Badan)_Rev.docx (7601 karakter)
✓ 18. Form SKJ Pengawas (Kasubag Kesekretariatan Sekretaris Utama)_Rev.docx (7606 karakter)
✓ 2. Draft_SKJ_Inspektur_alx_edit - Inspektur Utama v2.docx (11431 karakter)
✓ 22. Form SKJ Pengawas (Kasubag Kesekretariatan Deputi Bidang Penindakan)_Rev.docx (7664 karakter)
✓ 23. Form SKJ Administrator (Kabag Tata Usaha).docx (9403 karakt

In [59]:
# load_dotenv(dotenv_path=".env", override=True)
rag = await initialize_rag() 

INFO: PostgreSQL table: LIGHTRAG_VDB_ENTITY_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_RELATION_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_CHUNKS_text_embedding_3_small_1536d
INFO: PostgreSQL, Retry config: attempts=10, backoff=3.0s, backoff_max=30.0s, pool_close_timeout=5.0s
INFO: PostgreSQL, VECTOR extension enabled
INFO: PostgreSQL, Connected to database at 127.0.0.1:5452/lightrag without SSL
ERROR: PostgreSQL database, error:relation "lightrag_doc_full" does not exist
INFO: PostgreSQL, Try Creating table LIGHTRAG_DOC_FULL in database
INFO: PostgreSQL, Creation success table LIGHTRAG_DOC_FULL in PostgreSQL database
ERROR: PostgreSQL database, error:relation "lightrag_doc_chunks" does not exist
INFO: PostgreSQL, Try Creating table LIGHTRAG_DOC_CHUNKS in database
INFO: PostgreSQL, Creation success table LIGHTRAG_DOC_CHUNKS in PostgreSQL database
ERROR: PostgreSQL database, error:relation "lightrag_llm_cache" does not exist
INFO: PostgreSQL

In [60]:
# Rag initialization
rag = await initialize_rag() 

# Test embedding function
test_text = ["This is a test string for embedding."]
embedding = await rag.embedding_func(test_text)
embedding_dim = embedding.shape[1]
print("\n=======================")
print("Test embedding function")
print("========================")
print(f"Test dict: {test_text}")
print(f"Detected embedding dimension: {embedding_dim}\n\n")

INFO: PostgreSQL table: LIGHTRAG_VDB_ENTITY_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_RELATION_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_CHUNKS_text_embedding_3_small_1536d
INFO: HNSW vector index idx_c79820f98cd4_hnsw_cosine already exists on table LIGHTRAG_VDB_ENTITY_text_embedding_3_small_1536d
INFO: HNSW vector index idx_a6b494f856e1_hnsw_cosine already exists on table LIGHTRAG_VDB_RELATION_text_embedding_3_small_1536d
INFO: HNSW vector index idx_036611e7ab3b_hnsw_cosine already exists on table LIGHTRAG_VDB_CHUNKS_text_embedding_3_small_1536d
INFO: [base] Connected to neo4j at neo4j://127.0.0.1:7687
INFO: [base] Ensured B-Tree index on entity_id for base in neo4j
INFO: [base] Found existing index 'entity_id_fulltext_idx_base' with state: ONLINE
INFO: [base] Full-text index 'entity_id_fulltext_idx_base' already exists and is online. Skipping recreation.
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, 


Test embedding function
Test dict: ['This is a test string for embedding.']
Detected embedding dimension: 1536




In [61]:
teks_skj_full = read_folder("../data/skj_documents")


✓ 1. Form SKJ Pimpinan Tinggi Madya Sekretaris Utama (Level 5).docx (11372 karakter)
✓ 10. Draft_SKJ_Inspektur_alx_edit - Inspektur 2 v2.docx (11056 karakter)
✓ 11. Form SKJ JPT Pratama (Kepala Biro Umum)_rev1.docx (10188 karakter)
✓ 12. Form SKJ Administrator (Kabag PBJ)_rev1.docx (8224 karakter)
✓ 13. Form SKJ Administrator (Kabag PBMN)_rev1.docx (8572 karakter)
✓ 14. Form SKJ Administrator (Kabag Rumah Tangga)_rev1.docx (8781 karakter)
✓ 15. Form SKJ Administrator (Kabag PKP)_rev1.docx (8488 karakter)
✓ 16. Form SKJ Pengawas (Kasubag Protokol)_Rev.docx (7343 karakter)
✓ 17. Form SKJ Pengawas (Kasubag Kesekretariatan Kepala Badan)_Rev.docx (7601 karakter)
✓ 18. Form SKJ Pengawas (Kasubag Kesekretariatan Sekretaris Utama)_Rev.docx (7606 karakter)
✓ 2. Draft_SKJ_Inspektur_alx_edit - Inspektur Utama v2.docx (11431 karakter)
✓ 22. Form SKJ Pengawas (Kasubag Kesekretariatan Deputi Bidang Penindakan)_Rev.docx (7664 karakter)
✓ 23. Form SKJ Administrator (Kabag Tata Usaha).docx (9403 karakt

In [62]:
# Pdf use pdfplumber and docx use custom reader karena banyak tabel
# 1. Insert docs skj
await rag.ainsert(
    [doc["text"] for doc in teks_skj_full],
    file_paths=[doc["filename"] for doc in teks_skj_full]
)

INFO: Processing 20 document(s)
INFO: Extracting stage 1/20: 2. Draft_SKJ_Inspektur_alx_edit - Inspektur Utama v2.docx
INFO: Processing d-id: doc-4f52b942ba77c4c93c89fc28ebacb8a3
INFO: Extracting stage 2/20: 1. Form SKJ Pimpinan Tinggi Madya Sekretaris Utama (Level 5).docx
INFO: Processing d-id: doc-ee0f10f10c1e1b0bb0cec1963d448959
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: default:extract:6870694f9aab8f017e50681f1a622ffd
INFO:  == LLM cache == saving: default:extract:d99237ee9e97f5ca0e6501a28ad953af
INFO:  == LLM cache == saving: default:extract:eab97cd80d0f784507e09f8746686d6b
INFO:  == LLM cache == saving: default:extract:23cc61de7c3ce9e10694697e9d0cc5b1
INFO: Chunk 1 of 3 extracted 15 Ent + 14 Rel chunk-c8f71ba6a4640147fdd65dd8fa3f02e7
INFO:  == LLM cache == saving: default:extract:06155c6b831b619ae73d6c5d16822aa3
INFO: Chunk 1 of 3 extracted 13 Ent + 13 Rel chunk-f4af5b3bae3564e18502b0d7f2f221ea

'insert_20260423_132450_cc6ea0b6'

In [63]:
from lightrag import QueryParam
result = await rag.aquery(
    "Apa saja kompetensi yang dipersyaratkan untuk jabatan Sekretaris Utama?",
    param=QueryParam(mode="local")
)
print(result)

INFO:  == LLM cache == saving: local:keywords:20af5bed71b4113b2763edf59bbf1301
INFO: Query nodes: Kualifikasi pendidikan, Pengalaman kerja, Kemampuan manajerial, Sertifikasi, Keterampilan administrasi (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 308 relations
INFO: Raw search results: 40 entities, 308 relations, 0 vector chunks
INFO: After truncation: 30 entities, 127 relations
INFO: Selecting 44 from 44 entity-related chunks by vector similarity
INFO: Find 1 additional chunks in 1 relations (deduplicated 32)
INFO: Selecting 1 from 1 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 45 -> 45 (deduplicated 0)
INFO: Final context: 30 entities, 127 relations, 13 chunks
INFO: Final chunks S+F/O: E4/1 R1/1 E6/2 E5/3 E4/4 E4/5 E5/6 E3/7 E3/8 E4/9 E3/10 E3/11 E5/12
INFO:  == LLM cache == saving: local:query:b0c69e36c2d6c96e172db0e4c968f226


Berikut adalah kompetensi yang dipersyaratkan untuk jabatan Sekretaris Utama (Jabatan Pimpinan Tinggi Madya) berdasarkan standar kompetensi yang berlaku:

### Kompetensi Manajerial

1. **Integritas (Level 5)**
   - Mempertahankan standar keadilan dan etika tingkat tinggi, menjadi role model dalam penerapan etika di tingkat nasional, serta membuat kebijakan dan strategi integritas yang sesuai nilai strategis organisasi.

2. **Kerjasama (Level 5)**
   - Menciptakan situasi kerja sama secara konsisten, baik di dalam maupun di luar instansi, menjaga sinergi dengan pemangku kepentingan, membangun konsensus, dan meningkatkan produktivitas organisasi.

3. **Komunikasi (Level 5)**
   - Menggagas sistem komunikasi yang terbuka secara strategis, mampu menghilangkan hambatan komunikasi, dan menggunakan berbagai saluran di tingkat nasional untuk mencapai kesepakatan serta mencari solusi.

4. **Orientasi pada Hasil (Level 5)**
   - Memastikan mutu dan keberlanjutan hasil kerja organisasi, memastika

## ADD RAGanything

In [65]:
from raganything import RAGAnything
from lightrag.llm.openai import openai_embed
from lightrag.utils import EmbeddingFunc

In [66]:
# Reinitialize Lightrag with RAGAnything as vision model
rag = RAGAnything(
    lightrag=rag,
    vision_model_func=lambda prompt, system_prompt=None, history_messages=[], image_data=None, **kwargs: openai_complete_if_cache(
        "gpt-4o",
        "",
        system_prompt=None,
        history_messages=[],
        messages=[
            {"role": "system", "content": system_prompt} if system_prompt else None,
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}
            ]} if image_data else {"role": "user", "content": prompt}
        ],
        api_key=os.getenv("LLM_BINDING_API_KEY"),
        **kwargs,
    ) if image_data else openai_complete_if_cache(
        "gpt-4o-mini",
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("LLM_BINDING_API_KEY"),
        **kwargs,
    )
)

INFO: RAGAnything initialized with config:
INFO:   Working directory: ./rag_storage
INFO:   Parser: mineru
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1


In [72]:
result2 = await rag.aquery_with_multimodal(
    "What data has been processed in this LightRAG instance?",
    mode="hybrid"
)
print("Query result:", result2)

INFO: Executing multimodal query: What data has been processed in this LightRAG instance?...
INFO: Query mode: hybrid
INFO: No multimodal content provided, executing text query
INFO: Executing VLM enhanced query: What data has been processed in this LightRAG instance?...
INFO: Query nodes: LightRAG, Data terproses, Instans LightRAG (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 54 relations
INFO: Query edges: Data yang telah diproses, Instansiasi LightRAG (top_k:40, cosine:0.2)
INFO: Global query: 56 entites, 40 relations
INFO: Raw search results: 76 entities, 71 relations, 0 vector chunks
INFO: After truncation: 56 entities, 71 relations
INFO: Selecting 38 from 38 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 71 relations
INFO: Round-robin merged chunks: 38 -> 38 (deduplicated 0)
INFO: Final context: 56 entities, 71 relations, 19 chunks
INFO: Final chunks S+F/O: E2/1 E1/2 E4/3 E3/4 E5/5 E3/6 E11/7 E1/8 E11/9 E2/10 E1/11 E1/12 

Query result: Based on the available context, this LightRAG instance has processed a variety of data related to standards of competency for government positions within the organizational context of Indonesian public administration (such as BPOM). The data includes:

- Detailed standards of competency (standar kompetensi jabatan) and job descriptions for various positions (e.g., Administrator, Pengawas, Inspektur II, Kepala Subbagian, Kepala Bagian, etc.)
- Requirements for positions, including educational qualifications, experience, necessary managerial and technical training, and functional skills.
- Core managerial, social-cultural, and technical competencies, such as decision making, managing change, integrity, teamwork, public service, communication, and the technical aspects like office management, asset management, reporting, and follow-up on audit findings.
- Performance indicators (indikator kinerja jabatan), such as realization of budgets, timely completion of work unit perfor

In [ ]:
await rag.process_document_complete(
    file_path="../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf",
    output_dir="./output"
)

INFO: Starting complete document processing: ../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf
INFO: Starting document parsing: ../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf
INFO: Using mineru parser with method: auto
INFO: Detected PDF file, using parser for PDF...
[MinerU] Predict:   9%|▉         | 6/64 [58:29<9:30:23, 590.05s/it]Error: 1 task(s) failed while processing documents:
ERROR: Mineru command failed: Mineru command failed with return code 1: ['Predict:   9%|▉         | 6/64 [58:29<9:30:23, 590.05s/it]Error: 1 task(s) failed while processing documents:']


MineruExecutionError: Mineru command failed with return code 1: ['Predict:   9%|▉         | 6/64 [58:29<9:30:23, 590.05s/it]Error: 1 task(s) failed while processing documents:']

In [77]:
await rag.process_document_complete_lightrag_api(
    file_path="../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf",
    output_dir="./output"
)

INFO: Starting complete document processing: ../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf
INFO: Starting document parsing: ../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf
INFO: Using mineru parser with method: auto
INFO: Detected PDF file, using parser for PDF...
[MinerU] Predict:   9%|▉         | 6/64 [58:53<9:32:58, 592.74s/it]Error: 1 task(s) failed while processing documents:
ERROR: Mineru command failed: Mineru command failed with return code 1: ['Predict:   9%|▉         | 6/64 [58:53<9:32:58, 592.74s/it]Error: 1 task(s) failed while processing documents:']
INFO: Error processing document ../data/renstra_bpom/Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf: MineruExecutionError


False

In [79]:
#2
await rag.process_document_complete_lightrag_api(
    file_path="../data/renstra_bpom/Renstra Balai Besar POM di Jakarta Tahun 2025 2029.pdf",
    output_dir="./output"
)

INFO: Starting complete document processing: ../data/renstra_bpom/Renstra Balai Besar POM di Jakarta Tahun 2025 2029.pdf
INFO: Starting document parsing: ../data/renstra_bpom/Renstra Balai Besar POM di Jakarta Tahun 2025 2029.pdf
INFO: Using mineru parser with method: auto
INFO: Detected PDF file, using parser for PDF...


CancelledError: 